# 01 — Data Cleaning & Pipeline Version Builder

Produces all pipeline inputs for the 4-person team project.

| Section | What it does | Output |
|---------|-------------|--------|
| **Part 1 — Audit** | EDA: shape, dtypes, missing, skewness, correlation | `outputs/audit_report.json` |
| **Part 2 — Schema Cleaning** | Strip spaces, drop zero-variance, flag binary, stratified 70/15/15 split | `data/processed/` |
| **Part 3 — Version A** | StandardScaler · 94 features · natural imbalance | Role 2: Logistic Regression, Linear SVM |
| **Part 4 — Version B** | RobustScaler + SMOTE · 94 features · 50% balanced train | Role 2: XGBoost, Random Forest, LightGBM |
| **Part 5 — Version C** | StandardScaler + correlation pruning >0.95 · 74 features | Role 3: MLP (feature-reduced) |
| **Part 6 — Version D** | StandardScaler + PCA 95% variance · 50 components | Role 3: MLP, TabNet (compressed) |

**Run top-to-bottom.** All outputs are written to disk for downstream notebooks.

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
TARGET = "Bankrupt?"
ROOT          = Path("..")
RAW_PATH      = ROOT / "data" / "raw" / "data.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
OUTPUTS_DIR   = ROOT / "outputs"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print("Setup OK")

---
## Part 1 — Audit
Read the raw file and record dataset properties. Nothing is modified here.

In [ ]:
# Load raw data — strip all column name spaces immediately
df_raw = pd.read_csv(RAW_PATH)
df_raw.columns = df_raw.columns.str.strip()
print(f"Shape: {df_raw.shape}")
df_raw.head(3)

In [ ]:
# Dtype distribution
dtype_counts = df_raw.dtypes.astype(str).value_counts().to_dict()
print("Dtypes:", dtype_counts)

In [ ]:
# Missing values
missing_values = {c: int(n) for c, n in df_raw.isnull().sum().items() if n > 0}
print("Missing values:", missing_values if missing_values else "None")

In [ ]:
# Zero-variance columns (constant across all rows)
zero_var_cols = [c for c in df_raw.columns if df_raw[c].nunique() <= 1]
print("Zero-variance columns:", zero_var_cols)

In [ ]:
# Class balance
total = len(df_raw)
bankrupt_count = int(df_raw[TARGET].sum())
class_balance = {
    "total": total,
    "bankrupt_count": bankrupt_count,
    "solvent_count": total - bankrupt_count,
    "bankrupt_pct": round(bankrupt_count / total * 100, 4),
}
print(class_balance)

In [ ]:
# Top-10 most skewed numeric features (excluding target)
numeric_cols = [c for c in df_raw.select_dtypes(include=np.number).columns if c != TARGET]
skew_abs = df_raw[numeric_cols].skew().abs().sort_values(ascending=False)
top10_skewed = [
    {"column": col, "skewness": round(float(df_raw[col].skew()), 6)}
    for col in skew_abs.head(10).index
]
pd.DataFrame(top10_skewed)

In [ ]:
# Top-10 most correlated feature pairs (upper triangle, excluding target)
corr_abs = df_raw[numeric_cols].corr().abs()
upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))
stack = upper.stack().sort_values(ascending=False)

top10_corr = [
    {"col_a": a, "col_b": b, "correlation": round(float(df_raw[a].corr(df_raw[b])), 6)}
    for (a, b), _ in stack.head(10).items()
]
pd.DataFrame(top10_corr)

In [ ]:
# Save audit report
audit_report = {
    "shape": list(df_raw.shape),
    "dtypes": dtype_counts,
    "missing_values": missing_values,
    "zero_variance_columns": zero_var_cols,
    "class_balance": class_balance,
    "top10_most_skewed": top10_skewed,
    "top10_most_correlated_pairs": top10_corr,
}
with open(OUTPUTS_DIR / "audit_report.json", "w") as f:
    json.dump(audit_report, f, indent=2)
print(f"Saved → outputs/audit_report.json")

---
## Part 2 — Schema Cleaning & Stratified Split
Rules:
- Drop zero-variance columns before anything else
- Never fit any transformer before the split
- Stratified split ensures ~3.2% minority class in every fold

In [ ]:
# Drop zero-variance columns
df = df_raw.drop(columns=zero_var_cols)
print(f"Dropped: {zero_var_cols}")
print(f"Shape after drop: {df.shape}")

In [ ]:
# Flag binary columns (exactly 2 unique non-null values, excluding target)
binary_cols = sorted([
    c for c in df.columns
    if c != TARGET and df[c].dropna().nunique() == 2
])
print(f"Binary columns ({len(binary_cols)}): {binary_cols}")
(PROCESSED_DIR / "binary_columns.txt").write_text("\n".join(binary_cols) + "\n")
print("Saved → data/processed/binary_columns.txt")

In [ ]:
# Stratified 70 / 15 / 15 split
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

train_df = X_train.copy(); train_df[TARGET] = y_train
val_df   = X_val.copy();   val_df[TARGET]   = y_val
test_df  = X_test.copy();  test_df[TARGET]  = y_test

# Verify stratification
for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    n = len(split); b = split[TARGET].sum()
    print(f"{name:5s}: {n:5d} rows | bankrupt={int(b):3d} ({b/n*100:.2f}%)")

In [ ]:
# Save raw (unscaled) splits — source of truth for all versions
train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
val_df.to_csv(  PROCESSED_DIR / "val.csv",   index=False)
test_df.to_csv( PROCESSED_DIR / "test.csv",  index=False)
print("Saved → data/processed/train.csv / val.csv / test.csv")

In [ ]:
# Prepare numpy arrays for version building
feature_cols = [c for c in train_df.columns if c != TARGET]

Xtr = train_df[feature_cols].values;  ytr = train_df[TARGET].values
Xv  = val_df[feature_cols].values;    yv  = val_df[TARGET].values
Xte = test_df[feature_cols].values;   yte = test_df[TARGET].values

print(f"Feature count: {len(feature_cols)}")
print(f"Train: {Xtr.shape} | Val: {Xv.shape} | Test: {Xte.shape}")

---
## Part 3 — Version A: StandardScaler · No resampling
**For:** Logistic Regression, Linear SVM  
**Why:** These models assume zero-mean unit-variance features. Class imbalance is handled via `class_weight='balanced'` at training time — no synthetic data needed.  
**Features:** 94 (all original)

In [ ]:
ss = StandardScaler()
Xtr_A = ss.fit_transform(Xtr)   # fit on train only
Xv_A  = ss.transform(Xv)
Xte_A = ss.transform(Xte)

def to_df(X, y, cols):
    d = pd.DataFrame(X, columns=cols)
    d[TARGET] = y.astype(int)
    return d

to_df(Xtr_A, ytr, feature_cols).to_csv(OUTPUTS_DIR / "version_A_train.csv", index=False)
to_df(Xv_A,  yv,  feature_cols).to_csv(OUTPUTS_DIR / "version_A_val.csv",   index=False)
to_df(Xte_A, yte, feature_cols).to_csv(OUTPUTS_DIR / "version_A_test.csv",  index=False)

print(f"Version A | StandardScaler | no resampling")
print(f"  train: {len(Xtr_A)} rows, {Xtr_A.shape[1]} features, {int(ytr.sum())} bankrupt ({ytr.mean()*100:.2f}%)")
print(f"  val:   {len(Xv_A)} rows | test: {len(Xte_A)} rows")
print("  Saved → outputs/version_A_*.csv")

---
## Part 4 — Version B: RobustScaler + SMOTE
**For:** XGBoost, Random Forest, LightGBM  
**Why:** Tree-based models are not sensitive to scale but benefit from balanced training data. RobustScaler (uses median/IQR) reduces the influence of the extreme outliers seen in financial ratios (max skewness ~82). SMOTE generates synthetic minority samples in feature space, balancing train to 50/50.  
**Features:** 94 (all original) | **SMOTE on train only — val/test unchanged**

In [ ]:
rs = RobustScaler()
Xtr_B_sc = rs.fit_transform(Xtr)  # fit on train only
Xv_B_sc  = rs.transform(Xv)
Xte_B_sc = rs.transform(Xte)

# SMOTE on train only
Xtr_B, ytr_B = SMOTE(random_state=RANDOM_STATE, k_neighbors=5).fit_resample(Xtr_B_sc, ytr)

to_df(Xtr_B,   ytr_B, feature_cols).to_csv(OUTPUTS_DIR / "version_B_train.csv", index=False)
to_df(Xv_B_sc, yv,    feature_cols).to_csv(OUTPUTS_DIR / "version_B_val.csv",   index=False)
to_df(Xte_B_sc,yte,   feature_cols).to_csv(OUTPUTS_DIR / "version_B_test.csv",  index=False)

print(f"Version B | RobustScaler + SMOTE")
print(f"  train: {len(Xtr_B)} rows (original 4773), {int(ytr_B.sum())} bankrupt ({ytr_B.mean()*100:.2f}%)")
print(f"  val:   {len(Xv_B_sc)} rows (not resampled) | test: {len(Xte_B_sc)} rows (not resampled)")
print("  Saved → outputs/version_B_*.csv")

---
## Part 5 — Version C: StandardScaler + Correlation Pruning (>0.95)
**For:** MLP / Deep Learning  
**Why:** Neural networks don't benefit from redundant features — highly correlated inputs add noise without signal and slow convergence. We remove one column from every pair with |r| > 0.95 (computed on train only). StandardScaler is used because MLP gradient descent is sensitive to feature scale.  
**Features:** 94 → **74** (20 dropped)

In [ ]:
# Step 1: StandardScaler (fit on train only)
ss_C = StandardScaler()
Xtr_C_sc = ss_C.fit_transform(Xtr)
Xv_C_sc  = ss_C.transform(Xv)
Xte_C_sc = ss_C.transform(Xte)

# Step 2: Correlation pruning — computed on scaled train only
corr_matrix = pd.DataFrame(Xtr_C_sc, columns=feature_cols).corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
drop_cols = [c for c in upper_tri.columns if any(upper_tri[c] > 0.95)]
keep_cols = [c for c in feature_cols if c not in drop_cols]
keep_idx  = [feature_cols.index(c) for c in keep_cols]

Xtr_C = Xtr_C_sc[:, keep_idx]
Xv_C  = Xv_C_sc[:, keep_idx]
Xte_C = Xte_C_sc[:, keep_idx]

print(f"Correlation pruning threshold: |r| > 0.95")
print(f"  Dropped: {len(drop_cols)} features")
print(f"  Kept:    {len(keep_cols)} features")

to_df(Xtr_C, ytr, keep_cols).to_csv(OUTPUTS_DIR / "version_C_train.csv", index=False)
to_df(Xv_C,  yv,  keep_cols).to_csv(OUTPUTS_DIR / "version_C_val.csv",   index=False)
to_df(Xte_C, yte, keep_cols).to_csv(OUTPUTS_DIR / "version_C_test.csv",  index=False)

# Save column lists for reproducibility
(PROCESSED_DIR / "version_C_kept_columns.txt").write_text("\n".join(keep_cols) + "\n")
(PROCESSED_DIR / "version_C_dropped_columns.txt").write_text("\n".join(drop_cols) + "\n")

print(f"\nVersion C | StandardScaler + corr pruning")
print(f"  train: {len(Xtr_C)} rows, {Xtr_C.shape[1]} features, {int(ytr.sum())} bankrupt ({ytr.mean()*100:.2f}%)")
print("  Saved → outputs/version_C_*.csv")

---
## Part 6 — Version D: StandardScaler + PCA (95% variance)
**For:** MLP, TabNet  
**Why:** PCA compresses 94 correlated financial features into uncorrelated principal components, retaining 95% of the variance. This removes multicollinearity completely and reduces input dimensionality, which helps regularise MLP training and is a well-suited input format for TabNet's attention mechanism.  
**Features:** 94 → **50 PCA components** (95.46% variance retained)

In [ ]:
# Step 1: StandardScaler (fit on train only) — reuse ss_C or fit fresh
ss_D = StandardScaler()
Xtr_D_sc = ss_D.fit_transform(Xtr)
Xv_D_sc  = ss_D.transform(Xv)
Xte_D_sc = ss_D.transform(Xte)

# Step 2: PCA — fit on train only, 95% variance threshold
pca = PCA(n_components=0.95, random_state=RANDOM_STATE)
Xtr_D = pca.fit_transform(Xtr_D_sc)
Xv_D  = pca.transform(Xv_D_sc)
Xte_D = pca.transform(Xte_D_sc)

n_components     = pca.n_components_
explained_var    = pca.explained_variance_ratio_.sum() * 100
pca_cols         = [f"PC{i+1}" for i in range(n_components)]

print(f"PCA: {len(feature_cols)} features → {n_components} components ({explained_var:.2f}% variance)")

to_df(Xtr_D, ytr, pca_cols).to_csv(OUTPUTS_DIR / "version_D_train.csv", index=False)
to_df(Xv_D,  yv,  pca_cols).to_csv(OUTPUTS_DIR / "version_D_val.csv",   index=False)
to_df(Xte_D, yte, pca_cols).to_csv(OUTPUTS_DIR / "version_D_test.csv",  index=False)

# Save PCA metadata
pca_info = {
    "original_features": len(feature_cols),
    "n_components_95pct": int(n_components),
    "explained_variance_pct": round(float(explained_var), 4),
    "corr_pruning_threshold": 0.95,
    "version_C_kept_features": len(keep_cols),
    "version_C_dropped_features": len(drop_cols),
}
with open(OUTPUTS_DIR / "pca_info.json", "w") as f:
    json.dump(pca_info, f, indent=2)

print(f"\nVersion D | StandardScaler + PCA")
print(f"  train: {len(Xtr_D)} rows, {Xtr_D.shape[1]} components, {int(ytr.sum())} bankrupt ({ytr.mean()*100:.2f}%)")
print("  Saved → outputs/version_D_*.csv")
print("  Saved → outputs/pca_info.json")

---
## Part 7 — Version E: Feature Engineering + Aggressive Selection
**Source:** Friend's v2 pipeline, reproduced on our split for row-alignment  
**For:** Logistic Regression, interpretable/explainable ML  
**Why:** 12 domain-knowledge features engineered (profitability momentum, liquidity trend,
debt ratios, interaction terms), then filtered to features most correlated with bankruptcy.
Fewer, meaningful features → interpretable coefficients and cleaner SHAP values.  
**Features:** 94 + 12 engineered → **16 final** (corr pruning + low-var + target-corr filter)

In [ ]:
from sklearn.impute import SimpleImputer

def safe_divide(a, b, eps=1e-6):
    return a / (np.abs(b) + eps)

def add_features(df):
    df = df.copy()
    if 'ROA(A) before interest and % after tax' in df.columns:
        roa = df['ROA(A) before interest and % after tax']
        df['feat_roa_momentum'] = roa - roa.mean()
    if 'Liability to Equity' in df.columns:
        le = df['Liability to Equity']
        df['feat_leverage_growth'] = le - le.mean()
    if 'Current Ratio' in df.columns:
        cr = df['Current Ratio']
        df['feat_liquidity_trend'] = cr - cr.mean()
    if 'CFO to Assets' in df.columns and 'Liability to Equity' in df.columns:
        df['feat_debt_service_ratio'] = safe_divide(df['CFO to Assets'],
            df['Liability to Equity'] / (1 + df['Liability to Equity']))
    if 'ROA(A) before interest and % after tax' in df.columns and 'Debt ratio %' in df.columns:
        df['feat_prof_to_debt'] = safe_divide(
            df['ROA(A) before interest and % after tax'], df['Debt ratio %'])
    if 'Quick Ratio' in df.columns and 'Current Ratio' in df.columns:
        df['feat_asset_quality'] = safe_divide(df['Quick Ratio'], df['Current Ratio'])
    if 'Operating Profit Rate' in df.columns and 'Borrowing dependency' in df.columns:
        df['feat_profit_x_leverage'] = df['Operating Profit Rate'] * df['Borrowing dependency']
    if 'Current Ratio' in df.columns and 'Debt ratio %' in df.columns:
        df['feat_liquidity_x_debt'] = df['Current Ratio'] * df['Debt ratio %']
    if 'Cash Flow to Liability' in df.columns and 'Current Liabilities/Liability' in df.columns:
        df['feat_cashflow_x_liability'] = (
            df['Cash Flow to Liability'] * df['Current Liabilities/Liability'])
    if 'Quick Assets/Total Assets' in df.columns and 'Current Assets/Total Assets' in df.columns:
        df['feat_quick_to_current'] = safe_divide(
            df['Quick Assets/Total Assets'], df['Current Assets/Total Assets'])
    if 'CFO to Assets' in df.columns and 'Operating Funds to Liability' in df.columns:
        df['feat_cfo_to_operating'] = safe_divide(
            df['CFO to Assets'], df['Operating Funds to Liability'])
    if 'Debt ratio %' in df.columns and 'Current Ratio' in df.columns:
        df['feat_financial_volatility'] = df['Debt ratio %'] * (1 / (df['Current Ratio'] + 1e-6))
    return df

# Step 1: Feature engineering
X_tr_fe = add_features(train_df.drop(columns=[TARGET]))
X_v_fe  = add_features(val_df.drop(columns=[TARGET]))
X_te_fe = add_features(test_df.drop(columns=[TARGET]))
for dfx in [X_tr_fe, X_v_fe, X_te_fe]:
    dfx.replace([np.inf, -np.inf], np.nan, inplace=True)
print(f'After feature engineering: {X_tr_fe.shape[1]} features')

# Step 2: Impute median (fit on train only)
imp = SimpleImputer(strategy='median')
X_tr_imp = pd.DataFrame(imp.fit_transform(X_tr_fe), columns=X_tr_fe.columns)
X_v_imp  = pd.DataFrame(imp.transform(X_v_fe),      columns=X_v_fe.columns)
X_te_imp = pd.DataFrame(imp.transform(X_te_fe),     columns=X_te_fe.columns)

# Step 3: Outlier clipping 1-99% (fit on train only)
lo = X_tr_imp.quantile(0.01); hi = X_tr_imp.quantile(0.99)
X_tr_clip = X_tr_imp.clip(lo, hi, axis=1)
X_v_clip  = X_v_imp.clip(lo,  hi, axis=1)
X_te_clip = X_te_imp.clip(lo,  hi, axis=1)

# Step 4: Correlation pruning |r| > 0.95 (from train only)
corr_E = X_tr_clip.corr().abs()
upper_E = corr_E.where(np.triu(np.ones(corr_E.shape), k=1).astype(bool))
drop_corr_E = [c for c in upper_E.columns if any(upper_E[c] > 0.95)]
X_tr_nc = X_tr_clip.drop(columns=drop_corr_E)
X_v_nc  = X_v_clip.drop(columns=drop_corr_E)
X_te_nc = X_te_clip.drop(columns=drop_corr_E)
print(f'After corr pruning: {X_tr_nc.shape[1]} features (dropped {len(drop_corr_E)})')

# Step 5: Low-variance filter var < 0.01 (from train only)
low_var_E = X_tr_nc.var()[X_tr_nc.var() < 0.01].index.tolist()
X_tr_lv = X_tr_nc.drop(columns=low_var_E)
X_v_lv  = X_v_nc.drop(columns=low_var_E)
X_te_lv = X_te_nc.drop(columns=low_var_E)
print(f'After low-var filter: {X_tr_lv.shape[1]} features (dropped {len(low_var_E)})')

# Step 6: Target-correlation selection |corr| > 0.01 (from train only)
tgt_corr = X_tr_lv.apply(lambda col: abs(pd.Series(ytr).corr(col)))
selected_E = tgt_corr[tgt_corr > 0.01].index.tolist()
X_tr_sel = X_tr_lv[selected_E]
X_v_sel  = X_v_lv[selected_E]
X_te_sel = X_te_lv[selected_E]
print(f'After target-corr filter: {len(selected_E)} features selected')
print(tgt_corr[selected_E].sort_values(ascending=False).to_string())

# Step 7: StandardScaler (fit on train only)
ss_E = StandardScaler()
Xtr_E = pd.DataFrame(ss_E.fit_transform(X_tr_sel), columns=selected_E)
Xv_E  = pd.DataFrame(ss_E.transform(X_v_sel),      columns=selected_E)
Xte_E = pd.DataFrame(ss_E.transform(X_te_sel),     columns=selected_E)

# Save
to_df(Xtr_E.values, ytr, selected_E).to_csv(OUTPUTS_DIR / 'version_E_train.csv', index=False)
to_df(Xv_E.values,  yv,  selected_E).to_csv(OUTPUTS_DIR / 'version_E_val.csv',   index=False)
to_df(Xte_E.values, yte, selected_E).to_csv(OUTPUTS_DIR / 'version_E_test.csv',  index=False)
(PROCESSED_DIR / 'version_E_selected_columns.txt').write_text('\n'.join(selected_E) + '\n')

print(f'\nVersion E saved: {len(Xtr_E)} rows, {len(selected_E)} features')
print('Saved -> outputs/version_E_*.csv')

summary = pd.DataFrame([
    {'version': 'A', 'scaler': 'StandardScaler', 'resampling': 'none',
     'train_rows': len(Xtr_A), 'train_bankrupt_count': int(ytr.sum()),
     'train_bankrupt_pct': round(ytr.mean()*100, 4),
     'val_rows': len(Xv_A), 'test_rows': len(Xte_A),
     'feature_count': Xtr_A.shape[1], 'notes': 'Logistic Regression, Linear SVM'},
    {'version': 'B', 'scaler': 'RobustScaler', 'resampling': 'SMOTE',
     'train_rows': len(Xtr_B), 'train_bankrupt_count': int(ytr_B.sum()),
     'train_bankrupt_pct': round(ytr_B.mean()*100, 4),
     'val_rows': len(Xv_B_sc), 'test_rows': len(Xte_B_sc),
     'feature_count': Xtr_B.shape[1], 'notes': 'XGBoost, Random Forest, LightGBM'},
    {'version': 'C', 'scaler': 'StandardScaler', 'resampling': 'none',
     'train_rows': len(Xtr_C), 'train_bankrupt_count': int(ytr.sum()),
     'train_bankrupt_pct': round(ytr.mean()*100, 4),
     'val_rows': len(Xv_C), 'test_rows': len(Xte_C),
     'feature_count': Xtr_C.shape[1], 'notes': f'MLP corr pruning >0.95 ({len(drop_cols)} dropped)'},
    {'version': 'D', 'scaler': 'StandardScaler+PCA', 'resampling': 'none',
     'train_rows': len(Xtr_D), 'train_bankrupt_count': int(ytr.sum()),
     'train_bankrupt_pct': round(ytr.mean()*100, 4),
     'val_rows': len(Xv_D), 'test_rows': len(Xte_D),
     'feature_count': Xtr_D.shape[1], 'notes': f'MLP/TabNet PCA {n_components} components ({explained_var:.1f}% var)'},
    {'version': 'E', 'scaler': 'StandardScaler', 'resampling': 'none',
     'train_rows': len(Xtr_E), 'train_bankrupt_count': int(ytr.sum()),
     'train_bankrupt_pct': round(ytr.mean()*100, 4),
     'val_rows': len(Xv_E), 'test_rows': len(Xte_E),
     'feature_count': len(selected_E),
     'notes': f'LogReg/explainable {len(selected_E)} engineered+selected features (friend v2)'},
])

summary.to_csv(OUTPUTS_DIR / 'version_summary.csv', index=False)
print('Saved -> outputs/version_summary.csv\n')
summary

In [ ]:
summary = pd.DataFrame([
    {"version": "A", "scaler": "StandardScaler", "resampling": "none",
     "train_rows": len(Xtr_A), "train_bankrupt_count": int(ytr.sum()),
     "train_bankrupt_pct": round(ytr.mean()*100, 4),
     "val_rows": len(Xv_A), "test_rows": len(Xte_A),
     "feature_count": Xtr_A.shape[1], "notes": "Logistic Regression, Linear SVM"},
    {"version": "B", "scaler": "RobustScaler", "resampling": "SMOTE",
     "train_rows": len(Xtr_B), "train_bankrupt_count": int(ytr_B.sum()),
     "train_bankrupt_pct": round(ytr_B.mean()*100, 4),
     "val_rows": len(Xv_B_sc), "test_rows": len(Xte_B_sc),
     "feature_count": Xtr_B.shape[1], "notes": "XGBoost, Random Forest, LightGBM"},
    {"version": "C", "scaler": "StandardScaler", "resampling": "none",
     "train_rows": len(Xtr_C), "train_bankrupt_count": int(ytr.sum()),
     "train_bankrupt_pct": round(ytr.mean()*100, 4),
     "val_rows": len(Xv_C), "test_rows": len(Xte_C),
     "feature_count": Xtr_C.shape[1], "notes": f"MLP — corr pruning >0.95 ({len(drop_cols)} dropped)"},
    {"version": "D", "scaler": "StandardScaler+PCA", "resampling": "none",
     "train_rows": len(Xtr_D), "train_bankrupt_count": int(ytr.sum()),
     "train_bankrupt_pct": round(ytr.mean()*100, 4),
     "val_rows": len(Xv_D), "test_rows": len(Xte_D),
     "feature_count": Xtr_D.shape[1], "notes": f"MLP/TabNet — PCA {n_components} components ({explained_var:.1f}% var)"},
])

summary.to_csv(OUTPUTS_DIR / "version_summary.csv", index=False)
print("Saved → outputs/version_summary.csv\n")
summary

# Verify all expected outputs exist and are non-empty
expected = [
    PROCESSED_DIR / 'train.csv',
    PROCESSED_DIR / 'val.csv',
    PROCESSED_DIR / 'test.csv',
    PROCESSED_DIR / 'binary_columns.txt',
    PROCESSED_DIR / 'version_C_kept_columns.txt',
    PROCESSED_DIR / 'version_C_dropped_columns.txt',
    PROCESSED_DIR / 'version_E_selected_columns.txt',
    OUTPUTS_DIR / 'audit_report.json',
    OUTPUTS_DIR / 'pca_info.json',
    OUTPUTS_DIR / 'version_summary.csv',
    *[OUTPUTS_DIR / f'version_{v}_{s}.csv'
      for v in ['A', 'B', 'C', 'D', 'E'] for s in ['train', 'val', 'test']],
]

all_ok = True
for p in expected:
    exists = p.exists() and p.stat().st_size > 0
    status = 'OK ' if exists else 'MISSING'
    print(f'  {status}  {p.relative_to(ROOT)}')
    if not exists:
        all_ok = False

print()
print('All outputs present.' if all_ok else 'WARNING: some outputs are missing!')

In [ ]:
# Verify all expected outputs exist and are non-empty
expected = [
    PROCESSED_DIR / "train.csv",
    PROCESSED_DIR / "val.csv",
    PROCESSED_DIR / "test.csv",
    PROCESSED_DIR / "binary_columns.txt",
    PROCESSED_DIR / "version_C_kept_columns.txt",
    PROCESSED_DIR / "version_C_dropped_columns.txt",
    OUTPUTS_DIR / "audit_report.json",
    OUTPUTS_DIR / "pca_info.json",
    OUTPUTS_DIR / "version_summary.csv",
    *[OUTPUTS_DIR / f"version_{v}_{s}.csv"
      for v in ["A", "B", "C", "D"] for s in ["train", "val", "test"]],
]

all_ok = True
for p in expected:
    exists = p.exists() and p.stat().st_size > 0
    status = "OK " if exists else "MISSING"
    print(f"  {status}  {p.relative_to(ROOT)}")
    if not exists:
        all_ok = False

print()
print("All outputs present." if all_ok else "WARNING: some outputs are missing!")